# Task 8: Semantic Segmentation Pipeline Using Custom Dice Loss

## Objective

Build a U-Net-style semantic segmentation pipeline for pixel-wise image-mask prediction.

The implementation includes:

- Image/mask dataset pipeline
- Albumentations transformations
- U-Net encoder-decoder architecture
- Skip connections
- Custom Soft Dice Loss
- Binary Cross-Entropy
- Composite Dice + BCE loss
- Pixel-wise precision-recall analysis
- Boundary precision-recall visualization

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_curve

torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"

IMAGE_SIZE = 128
NUM_SAMPLES = 80


# ============================================================
# Synthetic segmentation dataset
# ============================================================

class SyntheticSegmentationDataset(Dataset):

    def __init__(self, n=80):

        self.images = []
        self.masks = []

        for _ in range(n):

            image = np.zeros(
                (IMAGE_SIZE, IMAGE_SIZE, 3),
                dtype=np.float32
            )

            mask = np.zeros(
                (IMAGE_SIZE, IMAGE_SIZE),
                dtype=np.float32
            )

            # Random circular object
            cx = np.random.randint(30, 98)
            cy = np.random.randint(30, 98)
            r = np.random.randint(12, 25)

            yy, xx = np.ogrid[
                :IMAGE_SIZE,
                :IMAGE_SIZE
            ]

            circle = (
                (xx - cx) ** 2
                + (yy - cy) ** 2
                <= r ** 2
            )

            mask[circle] = 1

            image[..., 0] = mask
            image += np.random.normal(
                0,
                0.08,
                image.shape
            )

            image = np.clip(
                image,
                0,
                1
            )

            self.images.append(image)
            self.masks.append(mask)

        self.transform = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(p=0.3)
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        image = self.images[idx]
        mask = self.masks[idx]

        transformed = self.transform(
            image=image,
            mask=mask
        )

        image = transformed["image"]
        mask = transformed["mask"]

        image = torch.tensor(
            image.transpose(2, 0, 1),
            dtype=torch.float32
        )

        mask = torch.tensor(
            mask,
            dtype=torch.float32
        ).unsqueeze(0)

        return image, mask


dataset = SyntheticSegmentationDataset(
    NUM_SAMPLES
)

loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True
)

print("Dataset size:", len(dataset))

# U-Net Architecture

The U-Net consists of an encoder that extracts spatial features and a decoder that progressively reconstructs the segmentation mask.

Skip connections transfer high-resolution information from the encoder to the corresponding decoder stage.

In [ ]:
class DoubleConv(nn.Module):

    def __init__(self, in_ch, out_ch):

        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_ch,
                out_ch,
                3,
                padding=1
            ),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_ch,
                out_ch,
                3,
                padding=1
            ),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.enc1 = DoubleConv(3, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(
            128,
            256
        )

        self.up3 = nn.ConvTranspose2d(
            256,
            128,
            2,
            stride=2
        )

        self.dec3 = DoubleConv(
            256,
            128
        )

        self.up2 = nn.ConvTranspose2d(
            128,
            64,
            2,
            stride=2
        )

        self.dec2 = DoubleConv(
            128,
            64
        )

        self.up1 = nn.ConvTranspose2d(
            64,
            32,
            2,
            stride=2
        )

        self.dec1 = DoubleConv(
            64,
            32
        )

        self.output = nn.Conv2d(
            32,
            1,
            1
        )

    def forward(self, x):

        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        b = self.bottleneck(
            self.pool(e3)
        )

        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.output(d1)


model = UNet().to(device)

print(
    "Parameters:",
    sum(p.numel() for p in model.parameters())
)

# Custom Soft Dice + Binary Cross-Entropy Loss

Dice loss measures the overlap between predicted and target masks.

Binary Cross-Entropy provides pixel-wise classification supervision.

The composite loss combines both objectives:

\[
L = L_{Dice}+L_{BCE}
\]

In [ ]:
def soft_dice_loss(
    logits,
    targets,
    eps=1e-6
):

    probs = torch.sigmoid(logits)

    probs = probs.reshape(
        probs.shape[0],
        -1
    )

    targets = targets.reshape(
        targets.shape[0],
        -1
    )

    intersection = (
        probs * targets
    ).sum(dim=1)

    dice = (
        2 * intersection + eps
    ) / (
        probs.sum(dim=1)
        + targets.sum(dim=1)
        + eps
    )

    return 1 - dice.mean()


def composite_loss(
    logits,
    targets,
    dice_weight=0.5
):

    dice = soft_dice_loss(
        logits,
        targets
    )

    bce = nn.functional.binary_cross_entropy_with_logits(
        logits,
        targets
    )

    return (
        dice_weight * dice
        + (1 - dice_weight) * bce
    )


# Test loss
images, masks = next(iter(loader))

images = images.to(device)
masks = masks.to(device)

logits = model(images)

loss = composite_loss(
    logits,
    masks
)

print("Input :", images.shape)
print("Output:", logits.shape)
print("Loss  :", loss.item())

# Training the Segmentation Model

The U-Net is trained using the custom composite Dice + BCE loss.

After training, the predicted probability maps are converted into pixel-wise predictions for evaluation.

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

EPOCHS = 5
history = []

for epoch in range(EPOCHS):

    model.train()
    epoch_loss = 0

    for images, masks in loader:

        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        logits = model(images)

        loss = composite_loss(
            logits,
            masks
        )

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    epoch_loss /= len(loader)

    history.append(epoch_loss)

    print(
        f"Epoch {epoch+1}/{EPOCHS} "
        f"- Loss: {epoch_loss:.4f}"
    )


plt.figure(figsize=(7, 4))
plt.plot(history)
plt.xlabel("Epoch")
plt.ylabel("Composite Loss")
plt.title("U-Net Training Loss")
plt.grid(True)
plt.show()

In [ ]:
# ============================================================
# Pixel-wise Precision-Recall Boundary Curve
# ============================================================

model.eval()

all_probs = []
all_targets = []

with torch.no_grad():

    for images, masks in loader:

        images = images.to(device)

        logits = model(images)

        probs = torch.sigmoid(
            logits
        ).cpu().numpy()

        all_probs.append(
            probs.reshape(-1)
        )

        all_targets.append(
            masks.numpy().reshape(-1)
        )

all_probs = np.concatenate(
    all_probs
)

all_targets = np.concatenate(
    all_targets
)

precision, recall, thresholds = (
    precision_recall_curve(
        all_targets,
        all_probs
    )
)

plt.figure(figsize=(7, 5))

plt.plot(
    recall,
    precision
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(
    "Pixel-wise Precision-Recall Curve"
)

plt.grid(True)
plt.show()


# ============================================================
# Visualize segmentation
# ============================================================

images, masks = next(iter(loader))

with torch.no_grad():

    predictions = torch.sigmoid(
        model(images.to(device))
    ).cpu()

idx = 0

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(
    images[idx].permute(1, 2, 0)
)
plt.title("Input")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(
    masks[idx, 0],
    cmap="gray"
)
plt.title("Ground Truth")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(
    predictions[idx, 0] > 0.5,
    cmap="gray"
)
plt.title("Predicted Mask")
plt.axis("off")

plt.tight_layout()
plt.show()

# Conclusion

A complete semantic segmentation pipeline was implemented using a custom U-Net architecture.

The pipeline included:

- Image and pixel-wise mask loading
- Albumentations-based augmentation
- Encoder-decoder U-Net architecture
- Skip connections
- Custom Soft Dice Loss
- Binary Cross-Entropy
- Composite Dice + BCE loss
- Pixel-wise precision-recall analysis
- Ground-truth and predicted-mask visualization

Dice Loss directly optimizes spatial overlap, while Binary Cross-Entropy provides pixel-level classification supervision. Combining both losses provides a useful objective for segmentation problems where accurate boundaries and foreground-background balance are important.

The precision-recall curve provides an empirical view of pixel-wise segmentation performance, while the visual comparison demonstrates how the U-Net reconstructs the target mask from the input image.